In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
N_SCENARIOS = 200
N_HOSPITALS = 3

rows = []
for s in range(N_SCENARIOS):
    amb_lat = np.random.uniform(36.75, 36.90)
    amb_lon = np.random.uniform(10.10, 10.30)
    
    hospitals = []
    for h in range(N_HOSPITALS):
        h_lat = np.random.uniform(36.75, 36.90)
        h_lon = np.random.uniform(10.10, 10.30)
        
        dist = np.sqrt((amb_lat - h_lat)**2 + (amb_lon - h_lon)**2) * 111
        score_m1   = np.random.uniform(0.5, 1.0)  # from module 1
        traffic    = np.random.uniform(0.1, 0.9)
        time_est   = dist / (60 * (1 - 0.5 * traffic)) * 60
        spec_match = np.random.randint(0, 2)
        
        hospitals.append({
            "scenario": s,
            "hospital_id": h,
            "score_m1": score_m1,
            "distance_km": dist,
            "estimated_time_min": time_est,
            "traffic_level": traffic,
            "specialization_match": spec_match,
        })
    
   
    costs = [
        0.4 * r["estimated_time_min"] / 30
        - 0.3 * r["score_m1"]
        - 0.2 * r["specialization_match"]
        + 0.1 * r["traffic_level"]
        for r in hospitals
    ]
    best_idx = np.argmin(costs)
    
    for i, r in enumerate(hospitals):
        r["target"] = 1 if i == best_idx else 0
        rows.append(r)

df = pd.DataFrame(rows)
df.to_csv("hospital_decision_dataset.csv", index=False)

## Model — PyTorch pointwise scorer

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import requests

# 1. MODEL DEFINITION
class HospitalScorer(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),  nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze()

model = HospitalScorer()

# 2. TRAIN ON SIMULATED DATA
np.random.seed(42)
FEATURES = ["score_m1", "distance_km", "estimated_time_min",
            "traffic_level", "specialization_match"]

rows, targets = [], []
for _ in range(100):
    amb_lat = np.random.uniform(36.75, 36.90)
    amb_lon = np.random.uniform(10.10, 10.30)
    hospitals_batch = []
    for _ in range(3):
        h_lat = np.random.uniform(36.75, 36.90)
        h_lon = np.random.uniform(10.10, 10.30)
        dist  = np.sqrt((amb_lat - h_lat)**2 + (amb_lon - h_lon)**2) * 111
        traffic   = np.random.uniform(0.1, 0.9)
        time_est  = dist / (60 * (1 - 0.5 * traffic)) * 60
        score_m1  = np.random.uniform(0.5, 1.0)
        spec      = np.random.randint(0, 2)
        hospitals_batch.append([score_m1, dist, time_est, traffic, spec])
    
    costs = [0.4*(r[2]/30) - 0.3*r[0] - 0.2*r[4] + 0.1*r[3] for r in hospitals_batch]
    best = np.argmin(costs)
    for i, r in enumerate(hospitals_batch):
        rows.append(r)
        targets.append(1.0 if i == best else 0.0)

X = torch.tensor(rows,    dtype=torch.float32)
y = torch.tensor(targets, dtype=torch.float32)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.BCELoss()
for epoch in range(50):
    preds = model(X)
    loss  = loss_fn(preds, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
print(f"Training done — final loss: {loss.item():.4f}")

# 3. DECIDE FUNCTION
def decide(hospitals_list, amb_lat, amb_lon):
    """
    hospitals_list: list of dicts, each with keys:
        score_m1, distance_km, estimated_time_min,
        traffic_level, specialization_match
    Returns: (best_index, confidence_score)
    """
    feature_matrix = torch.tensor(
        [[h[f] for f in FEATURES] for h in hospitals_list],
        dtype=torch.float32
    )
    with torch.no_grad():
        scores = model(feature_matrix)
    best_idx = scores.argmax().item()
    return best_idx, scores[best_idx].item()

# 4. ROUTING FUNCTION 
def get_route(amb_lat, amb_lon, hosp_lat, hosp_lon):
    url = (
        f"https://router.project-osrm.org/route/v1/driving/"
        f"{amb_lon},{amb_lat};{hosp_lon},{hosp_lat}"
        f"?overview=full&geometries=geojson&steps=true"
    )
    res   = requests.get(url).json()
    route = res["routes"][0]
    return {
        "distance_km":  round(route["distance"] / 1000, 2),
        "duration_min": round(route["duration"] / 60, 1),
        "geometry":     route["geometry"]["coordinates"],
        "steps": [
            s["maneuver"].get("instruction", s["maneuver"]["type"])
            for leg in route["legs"] for s in leg["steps"]
        ]
    }

# 5 TESTING :

#  Ambulance position
amb_lat, amb_lon = 36.8065, 10.1815

#  Hospital candidates (output from module 1 + derived features)
hospitals = [
    {
        "name": "Hôpital Charles Nicolle",
        "lat": 36.8190, "lon": 10.1660,
        "score_m1": 0.91, "distance_km": 3.2,
        "estimated_time_min": 6.1,
        "traffic_level": 0.3, "specialization_match": 1
    },
    {
        "name": "Hôpital La Rabta",
        "lat": 36.8320, "lon": 10.1700,
        "score_m1": 0.74, "distance_km": 7.5,
        "estimated_time_min": 15.0,
        "traffic_level": 0.6, "specialization_match": 1
    },
    {
        "name": "Hôpital Mongi Slim",
        "lat": 36.8580, "lon": 10.1900,
        "score_m1": 0.62, "distance_km": 4.1,
        "estimated_time_min": 9.5,
        "traffic_level": 0.4, "specialization_match": 0
    },
]

# Run decision model
best_idx, confidence = decide(hospitals, amb_lat, amb_lon)
chosen = hospitals[best_idx]
print(f"\nModel decision: {chosen['name']}")
print(f"Confidence: {confidence:.2f}")

# Get real road route
route = get_route(amb_lat, amb_lon, chosen["lat"], chosen["lon"])
print(f"Distance: {route['distance_km']} km")
print(f"ETA:      {route['duration_min']} min")
print(f"Steps:    {route['steps']}")

# Final output dict 
output = {
    "decision":       chosen["name"],
    "confidence":     confidence,
    "distance_km":    route["distance_km"],
    "duration_min":   route["duration_min"],
    "route_geometry": route["geometry"],   
    "turn_by_turn":   route["steps"],
}
print("\nFull output:", output)

Training done — final loss: 0.5870

Model decision: Hôpital Charles Nicolle
Confidence: 0.45
Distance: 2.33 km
ETA:      3.5 min
Steps:    ['depart', 'turn', 'turn', 'turn', 'fork', 'new name', 'fork', 'turn', 'roundabout', 'exit roundabout', 'turn', 'arrive']

Full output: {'decision': 'Hôpital Charles Nicolle', 'confidence': 0.4547395706176758, 'distance_km': 2.33, 'duration_min': 3.5, 'route_geometry': [[10.181621, 36.806522], [10.181417, 36.807247], [10.181372, 36.807436], [10.181355, 36.807628], [10.181382, 36.808011], [10.181361, 36.808041], [10.1813, 36.808106], [10.181268, 36.808126], [10.181132, 36.80823], [10.181084, 36.808292], [10.181072, 36.808327], [10.181068, 36.808346], [10.181064, 36.808365], [10.18106, 36.808388], [10.180949, 36.809007], [10.179985, 36.808911], [10.180021, 36.808667], [10.179968, 36.80856], [10.179916, 36.808513], [10.17981, 36.808489], [10.179747, 36.808487], [10.179634, 36.808514], [10.179476, 36.808582], [10.179426, 36.808618], [10.179293, 36.80873